In [2]:
!pip install pymongo sentence-transformers numpy pillow pytesseract opencv-python

  Using cached pymongo-4.17.0-cp313-cp313-macosx_11_0_arm64.whl.metadata (10 kB)
  Using cached pytesseract-0.3.13-py3-none-any.whl.metadata (11 kB)
  Using cached opencv_python-4.13.0.92-cp37-abi3-macosx_13_0_arm64.whl.metadata (19 kB)
  Using cached dnspython-2.8.0-py3-none-any.whl.metadata (5.7 kB)
Using cached pymongo-4.17.0-cp313-cp313-macosx_11_0_arm64.whl (985 kB)
Using cached dnspython-2.8.0-py3-none-any.whl (331 kB)
Using cached pytesseract-0.3.13-py3-none-any.whl (14 kB)
Using cached opencv_python-4.13.0.92-cp37-abi3-macosx_13_0_arm64.whl (46.2 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [pymongo]m3/4 [pymongo]n]hon]

[notice] A new release of pip is available: 25.2 -> 26.1
[notice] To update, run: pip install --upgrade pip


In [9]:
from pymongo import MongoClient

client = MongoClient("mongodb://localhost:27017")
db = client.semantic_search
collection = db.documents

## Load an Embedding Model (FREE & local)

In [10]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

## 5.3 Insert Documents with Embeddings

In [12]:
documents = [
    "Apple offers a 30 day refund policy",
    "Customers can return products within one month",
    "Shipping takes 3 to 5 business days",
    "Technical support is available 24/7"
]

collection.delete_many({})

for doc in documents:
    embedding = model.encode(doc).tolist()
    collection.insert_one({
        "text": doc,
        "embedding": embedding
    })

## 5.4 Semantic Search (Manual Cosine Similarity)

In [13]:
import numpy as np

def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def semantic_search(query, top_k=2):
    query_embedding = model.encode(query)

    results = []
    for doc in collection.find():
        score = cosine_similarity(
            query_embedding,
            np.array(doc["embedding"])
        )
        results.append((doc["text"], score))

    results.sort(key=lambda x: x[1], reverse=True)
    return results[:top_k]

In [14]:
semantic_search("How do I get a refund?")

[('Apple offers a 30 day refund policy', np.float64(0.5600299547609461)),
 ('Customers can return products within one month',
  np.float64(0.27786558301277325))]

In [1]:
print("-" * 60)


------------------------------------------------------------
